## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', 30)
RANDOM_STATE = 0

## 2. Carga de datos y train/test split


In [ ]:
DATA_PATH = 'airbnb-listings-extract.csv'

df = pd.read_csv(DATA_PATH, sep=';', decimal='.')
print('Filas:', df.shape[0], '| Columnas:', df.shape[1])

df_train_raw, df_test_raw = train_test_split(df, test_size=0.2, shuffle=True, random_state=RANDOM_STATE)
print('Train:', df_train_raw.shape[0], 'filas')
print('Test: ', df_test_raw.shape[0], 'filas')

Filas: 14780 | Columnas: 89
Train: 11824 filas
Test:  2956 filas


## 3. Exploración inicial (EDA)


In [ ]:
row = df_train_raw.iloc[0]
for col in df_train_raw.columns:
    val = str(row[col])
    if len(val) > 55:
        val = val[:55] + '...'
    print(f'{col:32s} | {val}')

ID                               | 5994463
Listing Url                      | https://www.airbnb.com/rooms/5994463
Scrape ID                        | 20170407214119
Last Scraped                     | 2017-04-08
Name                             | PISO ATOCHA- FLAT NEAR ATOCHA  .
Summary                          | Piso recién reformado cómoda habitación con baño, salón...
Space                            | Un piso muy cómodo en Jerónimos, una de las zonas mas t...
Description                      | Piso recién reformado cómoda habitación con baño, salón...
Experiences Offered              | none
Neighborhood Overview            | El barrio de Jeronimos es un enclave tranquilo y muy fa...
Notes                            | Facilitamos guía de eventos y rutas turísticas que pued...
Transit                          | Excelente ubicación a tan solo 5 minutos andando de las...
Access                           | La conexión a internet por fibra óptica 100 MB. Máxima ...
Interaction            

## 4. Selección de columnas


In [ ]:
DROP_COLS = [
    # URLs e imágenes
    'Listing Url', 'Thumbnail Url', 'Medium Url', 'Picture Url', 'XL Picture Url',
    'Host URL', 'Host Thumbnail Url', 'Host Picture Url',
    # IDs y metadatos de scraping
    'ID', 'Scrape ID', 'Calendar last Scraped', 'Host ID',
    # Texto libre largo
    'Name', 'Summary', 'Space', 'Description', 'Neighborhood Overview',
    'Notes', 'Transit', 'Access', 'Interaction', 'House Rules', 'Host About',
    # Datos del host (no del piso)
    'Host Name', 'Host Location', 'Host Neighbourhood',
    # Geografía redundante
    'City', 'State', 'Market', 'Smart Location', 'Country', 'Country Code',
    'Zipcode', 'Street', 'Neighbourhood', 'Neighbourhood Group Cleansed',
    'Geolocation',
    # Casi vacías / constantes
    'Has Availability', 'License', 'Jurisdiction Names', 'Square Feet',
    'Experiences Offered', 'Calendar Updated',
    # Fuga de información (derivadas directamente de Price)
    'Weekly Price', 'Monthly Price',
    # Muy pocos datos reales (>97% vacío)
    'Host Acceptance Rate',
    # Fechas que se sustituyen por variables derivadas más adelante
    'First Review', 'Last Review',
]

df_work = df_train_raw.drop(columns=DROP_COLS)
print('Columnas restantes:', df_work.shape[1])

Columnas restantes: 41


## 5. Tratamiento de valores nulos


In [ ]:
missing_pct = (100 * df_work.isna().sum() / len(df_work)).sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

Security Deposit                  57.273342
Cleaning Fee                      41.018268
Review Scores Value               22.767253
Review Scores Location            22.758796
Review Scores Checkin             22.733424
Review Scores Accuracy            22.674222
Review Scores Communication       22.640392
Review Scores Cleanliness         22.631935
Review Scores Rating              22.505074
Reviews per Month                 21.473275
Host Response Time                12.745264
Host Response Rate                12.745264
Amenities                          1.192490
Bathrooms                          0.372124
Beds                               0.312923
Bedrooms                           0.169147
Price                              0.126861
Host Verifications                 0.059202
Calculated host listings count     0.033829
Host Since                         0.025372
Host Listings Count                0.025372
Host Total Listings Count          0.025372
Features                        

## 6. Ingeniería de variables

- `Host Since` (fecha) → `Years Being Host`: calculado **fila a fila** contra
  el `Last Scraped` de esa misma fila (no un año fijo — `Last Scraped` varía
  entre 2016 y 2017 según el anuncio, así que un año fijo sería impreciso).
- `Amenities`, `Host Verifications`, `Features` (listas de texto separadas
  por comas) → se cuentan los elementos (`N Amenities`, etc.) en vez de
  dejarlas como texto libre.

In [ ]:
def count_items(x):
    if pd.isna(x) or x in ('Unknown', ''):
        return 0
    return len(str(x).split(','))

df_work['Last Scraped'] = pd.to_datetime(df_work['Last Scraped'])
df_work['Host Since'] = pd.to_datetime(df_work['Host Since'])
df_work['Years Being Host'] = (df_work['Last Scraped'] - df_work['Host Since']).dt.days / 365
df_work = df_work.drop(columns=['Last Scraped', 'Host Since'])

for c in ['Amenities', 'Host Verifications', 'Features']:
    df_work[c] = df_work[c].fillna('Unknown')

df_work['N Amenities'] = df_work['Amenities'].apply(count_items)
df_work['N Verifications'] = df_work['Host Verifications'].apply(count_items)
df_work['N Features'] = df_work['Features'].apply(count_items)
df_work = df_work.drop(columns=['Amenities', 'Host Verifications', 'Features'])

print(df_work[['Years Being Host', 'N Amenities', 'N Verifications', 'N Features']].describe())

       Years Being Host   N Amenities  N Verifications    N Features
count      11821.000000  11824.000000     11824.000000  11824.000000
mean           2.554365     14.254482         4.116373      3.734016
std            1.612189      4.877951         1.281554      1.050629
min            0.002740      0.000000         0.000000      0.000000
25%            1.238356     11.000000         3.000000      3.000000
50%            2.315068     14.000000         4.000000      4.000000
75%            3.810959     17.000000         5.000000      4.000000
max            7.898630     39.000000        10.000000      8.000000


## 7. Codificación de variables categóricas

In [ ]:
low_card_cols = ['Property Type', 'Room Type', 'Bed Type', 'Cancellation Policy', 'Host Response Time']
df_work['Host Response Time'] = df_work['Host Response Time'].fillna('Unknown')

print('Categorías por columna:')
for c in low_card_cols + ['Neighbourhood Cleansed']:
    print(f'  {c}: {df_work[c].nunique()} valores distintos')

counts_barrio = df_work['Neighbourhood Cleansed'].value_counts()
print()
print('Barrios con 1 sola muestra:', (counts_barrio == 1).sum(), 'de', len(counts_barrio))

Categorías por columna:
  Property Type: 21 valores distintos
  Room Type: 3 valores distintos
  Bed Type: 5 valores distintos
  Cancellation Policy: 8 valores distintos
  Host Response Time: 5 valores distintos
  Neighbourhood Cleansed: 444 valores distintos

Barrios con 1 sola muestra: 146 de 444


## 8. Filtrado de outliers de `Price`


In [ ]:
print(df_work['Price'].describe())
print()
print('% de filas con Price > 200:', round(100*(df_work['Price']>200).mean(), 2), '%')

count    11809.000000
mean        73.712592
std         71.624844
min          9.000000
25%         34.000000
50%         55.000000
75%         87.000000
max        969.000000
Name: Price, dtype: float64

% de filas con Price > 200: 3.92 %


## 9. Funciones de preprocesado reutilizables


In [ ]:
def fit_preprocessing(df_train_raw, m_smoothing=20, price_cap=200):
    d = basic_clean(df_train_raw)
    artifacts = {}

    medians = {c: d[c].median() for c in MEDIAN_COLS}
    for c, med in medians.items():
        d[c] = d[c].fillna(med)
    artifacts['medians'] = medians

    d = pd.get_dummies(d, columns=LOW_CARD_COLS, drop_first=True)

    global_mean = d['Price'].mean()
    stats = d.groupby('Neighbourhood Cleansed')['Price'].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + m_smoothing * global_mean) / (stats['count'] + m_smoothing)
    artifacts['global_mean'] = global_mean
    artifacts['neighbourhood_map'] = smoothed

    d['Neighbourhood Encoded'] = d['Neighbourhood Cleansed'].map(smoothed)
    d = d.drop(columns=['Neighbourhood Cleansed'])

    d = d[d['Price'] <= price_cap]

    artifacts['feature_columns'] = [c for c in d.columns if c != 'Price']
    artifacts['price_cap'] = price_cap

    X = d[artifacts['feature_columns']]
    y = d['Price']
    return X, y, artifacts


def transform_test(df_test_raw, artifacts):
    d = basic_clean(df_test_raw)

    for c, med in artifacts['medians'].items():
        d[c] = d[c].fillna(med)

    d = pd.get_dummies(d, columns=LOW_CARD_COLS, drop_first=True)

    d['Neighbourhood Encoded'] = d['Neighbourhood Cleansed'].map(artifacts['neighbourhood_map'])
    d['Neighbourhood Encoded'] = d['Neighbourhood Encoded'].fillna(artifacts['global_mean'])
    d = d.drop(columns=['Neighbourhood Cleansed'])

    d = d[d['Price'] <= artifacts['price_cap']]

    y = d['Price']
    X = d.reindex(columns=artifacts['feature_columns'], fill_value=0)
    return X, y


X_train, y_train, artifacts = fit_preprocessing(df_train_raw)
X_test, y_test = transform_test(df_test_raw, artifacts)

assert list(X_train.columns) == list(X_test.columns)
print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Columnas alineadas correctamente entre train y test: OK')

X_train: (11345, 71)
X_test:  (2846, 71)
Columnas alineadas correctamente entre train y test: OK


## 10. Comparación de modelos con cross-validation


In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(random_state=RANDOM_STATE))
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)

resultados = {}
for nombre, modelo in [('Ridge', ridge), ('Random Forest', rf)]:
    scores = cross_val_score(modelo, X_train, y_train, cv=5,
                              scoring='neg_root_mean_squared_error', n_jobs=1)
    rmse = -scores
    resultados[nombre] = rmse.mean()
    print(f'{nombre:15s} | RMSE medio: {rmse.mean():.2f} EUR  (+/- {rmse.std():.2f})')

print()
print('Precio medio en train:', round(y_train.mean(), 1), 'EUR')

Ridge           | RMSE medio: 23.86 EUR  (+/- 0.83)
Random Forest   | RMSE medio: 20.31 EUR  (+/- 0.48)

Precio medio en train: 62.8 EUR


## 11. Ajuste de hiperparámetros (Random Forest)


In [ ]:
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, 20],
    'min_samples_leaf': [1, 5],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid, cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=1, verbose=1
)
grid.fit(X_train, y_train)

print()
print('Mejores hiperparámetros:', grid.best_params_)
print('Mejor RMSE (cross-validation):', round(-grid.best_score_, 2), 'EUR')

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Mejores hiperparámetros: {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 100}
Mejor RMSE (cross-validation): 20.56 EUR


## 12. Entrenamiento final y evaluación en test


In [ ]:
modelo_final = RandomForestRegressor(random_state=RANDOM_STATE, **grid.best_params_)
modelo_final.fit(X_train, y_train)

y_pred = modelo_final.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)
rmse_cv = -grid.best_score_

print(f'RMSE en TEST: {rmse_test:.2f} EUR')
print(f'MAE en TEST:  {mae_test:.2f} EUR')
print()
print(f'RMSE estimado por cross-validation en train: {rmse_cv:.2f} EUR')
print(f'Diferencia: {abs(rmse_test - rmse_cv):.2f} EUR -> el modelo generaliza bien, no hay sobreajuste ni fuga de datos')

RMSE en TEST: 20.46 EUR
MAE en TEST:  13.76 EUR

RMSE estimado por cross-validation en train: 20.56 EUR
Diferencia: 0.10 EUR -> el modelo generaliza bien, no hay sobreajuste ni fuga de datos


## 15. Conclusión
El modelo Random Forest, con un RMSE de 20.46 euros, predice el precio de alojamientos AIRBNB en Madrid con un error medio de entre 14 y 20 euros por noche. No obstante, este modelo solo es aplicable a alojamientos de hasta 200 euros por noche (en este caso, el 96% del dataset original). No se ha evaluado ningun alojamiento por encima de esa cifra.
El error del test (20.46 euros) es casi igual al recopilado por cross-validation en el train file (20.56 euros). Esto significa que el modelo es capaz de generalizar y no se escapan datos del train file al test file.
Se elige un modelo Random Forest porque, en una comparacion inicial frente a un modelo lineal, el RMSE es mejor. No obstante, al ajustar los hiper parámetros del random forest no se consigue mejorar el modelo, sugiriendo que los parametros 1) no son los adecuados o 2) que la configuracion del primer random forest era la mas adecuada para este caso.
